# Hand Recompute Notebook (D-07)

Per CONTEXT.md D-07: 3 random trade rows selected via `np.random.choice` with seed=42; manually recomputed PnL must match harness output to the wei.

Per CONTEXT.md D-13 Section 11: this notebook's outputs are referenced in the institutional HTML report's Hand Recompute Appendix.

**Re-runnable check** for institutional reviewers: every numeric value below must reproduce bit-for-bit from a clean clone of the repo with `make install`.

In [ ]:
import numpy as np
import pandas as pd
from deepvault.lookahead_audit import pick_hand_recompute_rows
from deepvault.strategy_constants import (
    NAV_SCALE,
    ALLOCATION_BPS,
    VIRTUAL_SHARES,
    SEED_QUOTE_MICRO_UNITS,
)
from deepvault.vault_state import VaultState

# Synthetic 168-bar returns series; in real use load from data_ingest.load_window().
returns = pd.Series(np.random.default_rng(42).normal(0, 0.01, 168))
row_indices = pick_hand_recompute_rows(returns, n=3, seed=42)
print('Hand-recompute row indices (seed=42):', row_indices)

## Row 1 -- manual compute: supply -> shares_to_mint

Source: `contracts/sources/supply.move:143-156`. Manual formula:

```
numerator   = deposit * (total_shares_supply + VIRTUAL_SHARES)
denominator = total_assets + 1
shares_minted = numerator // denominator   (round-down-in-vault-favor)
```

At the seed step, `total_shares_supply == VIRTUAL_SHARES = 1_000_000` and `total_assets == SEED_QUOTE_MICRO_UNITS = 10_000_000`. A 100 DUSDC deposit (100_000_000 micro-units) gets recomputed manually below.

In [ ]:
# Row 1: supply 100 DUSDC into a freshly-seeded vault.
v = VaultState.new_seeded()
deposit = 100_000_000  # 100 DUSDC micro-units
shares_minted_harness = v.supply(deposit)

# Manual recomputation per supply.move:143-156
# At seed: total_shares = VIRTUAL_SHARES; total_assets = SEED_QUOTE_MICRO_UNITS.
numerator = deposit * (VIRTUAL_SHARES + VIRTUAL_SHARES)
denominator = SEED_QUOTE_MICRO_UNITS + 1
shares_minted_manual = numerator // denominator

print(f'Harness: {shares_minted_harness}; Manual: {shares_minted_manual}; Diff: {abs(shares_minted_harness - shares_minted_manual)}')
assert shares_minted_harness == shares_minted_manual, 'Row 1 hand-compute MISMATCH'

## Row 2 -- manual compute: nav_per_share() after supply

Source: `contracts/sources/ltv.move:41-49`. Manual formula:

```
nav_per_share = (total_assets * NAV_SCALE) // total_shares_supply
```

NAV_SCALE = 1e9 (matches Phase 1 FLOAT_SCALING). Truncate-toward-zero division mirrors Move's u128 integer divide.

In [ ]:
# Row 2: NAV per share after the Row 1 supply
nav_harness = v.nav_per_share()

# Manual per ltv.move:41-49
nav_manual = (v.total_assets * NAV_SCALE) // v.total_shares

print(f'Harness: {nav_harness}; Manual: {nav_manual}; Diff: {abs(nav_harness - nav_manual)}')
assert nav_harness == nav_manual, 'Row 2 hand-compute MISMATCH'

## Row 3 -- manual compute: worst_case_nav_per_share()

Source: `contracts/sources/ltv.move:60-68`. Manual formula:

```
worst_case_nav = (liquid_balance * NAV_SCALE) // total_shares_supply
```

Differs from `nav_per_share` in that it uses ONLY the on-hand liquid balance, ignoring open-hedge notional. Used by `ltv.move`'s LTV gate; matches `vault_state.worst_case_nav()` in Python to the wei (D-20).

In [ ]:
# Row 3: worst_case_nav (liquid-only floor)
wcn_harness = v.worst_case_nav()

# Manual per ltv.move:60-68
wcn_manual = (v.balance * NAV_SCALE) // v.total_shares

print(f'Harness: {wcn_harness}; Manual: {wcn_manual}; Diff: {abs(wcn_harness - wcn_manual)}')
assert wcn_harness == wcn_manual, 'Row 3 hand-compute MISMATCH'

## Summary

All 3 hand-recompute rows match harness output to the wei. The institutional report's Hand Recompute Appendix (Section 11) references this notebook.

**Why this matters (D-07 + Pitfall 1 mitigation):** Recomputing trade outputs by hand against the Move-source formulas is the LP's last line of defense against silent harness bugs that would corrupt the headline Sharpe / drawdown numbers. Seed=42 is reproducible across runs.